# Pydantic model to structure output

TODO: Skapa en agent som ska simulera en IT-anställd 

In [ ]:
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

employee_simulator_agent = Agent(
    "openrouter:liquid/lfm-2.5-1.2b-thinking:free",
    system_prompt="""
    You are an HR expert within IT field in sweden within data science, data engineering, machine lerning, AI engineerin. You will simulate IT employees.
    
    Fields to include: 
    - Name
    - Age
    - Gender
    - Job_title 
    - Salary in SEK per month 
    """,)


result = await employee_simulator_agent.run("Simulate two employees")


In [3]:
print(result.output)

Here are two simulated IT employees in Sweden:

---

**Employee 1**  
- **Name**: Anna Larsen  
- **Age**: 34  
- **Gender**: Female  
- **Job_title**: Software Engineer  
- **Salary**: 620,000 SEK/month  

**Employee 2**  
- **Name**: Erikström Jóhannsson  
- **Age**: 41  
- **Gender**: Male  
- **Job_title**: Data Analyst  
- **Salary**: 750,000 SEK/month  

---

Let me know if you'd like adjustments or additional details!


In [4]:
with open("simulated_emlpoyees", "w") as file:
    file.write(result.output)


## Get more structured output 

issue with above: 
- output strycture vary 
- hard to work with the data e.g compute mean of salaries 

want: 
- repeatable struckture 

In [17]:

from pydantic import BaseModel, Field
from typing import Literal
from pydantic_ai import Agent


class EmployeeModel(BaseModel):
    name: str = Field(
        description="Mostly swedish names, but could be foreign names as well"
    )
    age: int = Field(description="age should be between 18 and 67")
    gender: Literal["Male", "Female"]
    experience_level: Literal["Entry", "Mid level", "Senior", "Expert"]
    job_title: str
    salary: int = Field(
        gte=30_000,
        lte=50_000,
        description="salary should be between 30k and 50k, the higher experience level, the higher salary",
    )


employee_simulator_agent = Agent(
    "openrouter:openai/gpt-oss-20b:free",
    system_prompt="""
You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.
""",
)

result = await employee_simulator_agent.run("Give me 3 employees", output_type=EmployeeModel)
result

/var/folders/nv/k5mh27wn1tlg3d6h94btf7v00000gn/T/ipykernel_73484/1413229117.py:14: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'gte', 'lte'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  salary: int = Field(


AgentRunResult(output=EmployeeModel(name='Erik Skärsmyr', age=34, gender='Male', experience_level='Mid level', job_title='Data Engineer', salary=42000))

In [8]:
result.output

EmployeeModel(name='Erik Åström', age=35, gender='Male', experience_level='Senior', job_title='Senior Data Scientist', salary=48000)

In [10]:
result.output.salary+5000

53000

In [18]:
employee_simulator_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt="""
You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.
""", retries=1
)

result = await employee_simulator_agent.run("Give me 3 employees", output_type=list[EmployeeModel], )
result

AgentRunResult(output=[EmployeeModel(name='Anna', age=28, gender='Female', experience_level='Mid level', job_title='Machine Learning Engineer', salary=35000), EmployeeModel(name='Erik', age=35, gender='Male', experience_level='Senior', job_title='Data Scientist', salary=42000), EmployeeModel(name='Sofia', age=42, gender='Female', experience_level='Expert', job_title='AI Architect', salary=48000)])

In [19]:
result.output[0].model_dump()

{'name': 'Anna',
 'age': 28,
 'gender': 'Female',
 'experience_level': 'Mid level',
 'job_title': 'Machine Learning Engineer',
 'salary': 35000}

TODO: 
- result.output make into list of dicts 
- create oandas dataframe based on this list 
- export a csv file of our simulates employees

In [20]:
employees = result.output
employees

[EmployeeModel(name='Anna', age=28, gender='Female', experience_level='Mid level', job_title='Machine Learning Engineer', salary=35000),
 EmployeeModel(name='Erik', age=35, gender='Male', experience_level='Senior', job_title='Data Scientist', salary=42000),
 EmployeeModel(name='Sofia', age=42, gender='Female', experience_level='Expert', job_title='AI Architect', salary=48000)]

In [23]:
employee_list = [employees.model_dump() for employees in employees]
employee_list 

[{'name': 'Anna',
  'age': 28,
  'gender': 'Female',
  'experience_level': 'Mid level',
  'job_title': 'Machine Learning Engineer',
  'salary': 35000},
 {'name': 'Erik',
  'age': 35,
  'gender': 'Male',
  'experience_level': 'Senior',
  'job_title': 'Data Scientist',
  'salary': 42000},
 {'name': 'Sofia',
  'age': 42,
  'gender': 'Female',
  'experience_level': 'Expert',
  'job_title': 'AI Architect',
  'salary': 48000}]

In [25]:
import pandas as pd 
df = pd.DataFrame(employee_list)
df 

,name,age,gender,experience_level,job_title,salary
0,Anna,28,Female,Mid level,Machine Learning Engineer,35000
1,Erik,35,Male,Senior,Data Scientist,42000
2,Sofia,42,Female,Expert,AI Architect,48000


In [26]:
df["salary"].mean()

np.float64(41666.666666666664)

In [28]:
df.to_csv("simulated_emlpoyees.csv", index=False)